In [1]:
import pandas as pd
import torch
data_folder = "/biodata/nyanovsky/datasets/dti/processed/v2/"
nodes = pd.read_csv(f"{data_folder}node_df.csv", index_col="node_id")
edges = pd.read_csv(f"{data_folder}edge_df_dedup.csv")

tensor_df = pd.read_csv(f"{data_folder}dti_tensor_df.csv", index_col=0)


full_dataset = torch.load(f"{data_folder}dti_full_dataset.pt")

In [2]:
nodes.head()

,ChG_deg,ChCh_deg,GG_deg,total_deg,node_type,louvain community (nodetype subgraph)
node_id,,,,,,
C155831,1,4,0,5,chem,141
G5243,41,0,17,58,gene,42
C24762158,9,2,0,11,chem,489
G213,85,0,3,88,gene,2
G506,6,0,3,9,gene,93


In [3]:
edges.head()

,src_id,trgt_id,edge_type,src_node_type,trgt_node_type,src_node_index,trgt_node_index
0,C155831,G5243,chg,chem,gene,0,1
1,C24762158,G213,chg,chem,gene,2,3
2,C24762158,G506,chg,chem,gene,2,4
3,C24762158,G563,chg,chem,gene,2,5
4,C24762158,G13884,chg,chem,gene,2,6


In [4]:
import sys
sys.path.append("..")
from models.training_utils import load_feature_dict
from data_class import GNNData

In [5]:
data = GNNData(node_path="/biodata/nyanovsky/datasets/dti/processed/v2/node_df.csv",
               edge_path="/biodata/nyanovsky/datasets/dti/processed/v2/edge_df_dedup.csv",
               node_index_col="node_id",
               src_edge_col="src_id",
               dst_edge_col="trgt_id",
               node_type_col="node_type",
               edge_type_col="edge_type",
               src_node_type_col="src_node_type",
               dst_node_type_col="trgt_node_type")

Feture initialization already worked okay. If unsure, test it by checking that the features were assigned correctly to their nodes using the node_mapping attribute (i.e, get feature for node "A", and check if feature for index node_mapping["A"] is equal to it)

In [6]:
gene_feature_dict = load_feature_dict(data_folder+"prot_features_64.txt", data_folder+"prot_features_ids.txt", 
                                                    tensor_df, "gene")
data.initialize_features(64, gene_feature_dict, inplace=True)

/usr/users/nyanovsky/tesis/gnn_interface/data_class.py:171: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  data_object.data[nodetype].x[tensor_idxs] = nodetype_embs


Check if PyG gets degrees right (against degrees that came from NetworkX)

In [7]:
print(data.data["gene"].degrees_by_edge_type[("chem", "chg", "gene")][:10])
print(data.data["chem"].degrees_by_edge_type[("chem", "chg", "gene")][:10])


tensor([41., 85.,  6.,  4.,  1.,  2.,  3.,  8., 19.,  3.])
tensor([1., 9., 3., 8., 1., 1., 2., 1., 2., 2.])


In [8]:
print(full_dataset["gene"]["degree_chg"][:10])
print(full_dataset["chem"]["degree_chg"][:10])
# Old dataset, degrees came from nx

[41 85  6  4  1  2  3  8 19  3]
[1 9 3 8 1 1 2 1 2 2]


In [9]:
from models.training_utils import NegativeSampler 

supervision_edge_type = ("chem", "chg", "gene")
src_degs = data.data["chem"].degrees_by_edge_type[supervision_edge_type]
dst_degs = data.data["gene"].degrees_by_edge_type[supervision_edge_type]

sampler = NegativeSampler(data.data, supervision_edge_type, src_degs, dst_degs)

In [10]:
from torch_geometric import seed_everything
seed_everything(42)
train_data, val_data, test_data = data.split_data(task="link_prediction",
                                                  negative_sampler=sampler,
                                                  supervision_edge_type=supervision_edge_type,
                                                  disjoint_train_ratio=0.2, 
                                                  num_val=0.1,
                                                  num_test=0.1)


/usr/users/nyanovsky/tesis/gnn_interface/../models/training_utils.py:183: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  probs = torch.tensor(self.weights[src_or_dst]**0.75)


Test if split works as expected

In [11]:
def test_equal_num_edges(dataset, sup_edge_type):
    num_chg_r = dataset[sup_edge_type
                        ]["edge_index"].shape[1]
    num_chg_l = dataset[(sup_edge_type[2], "rev_"+sup_edge_type[1], sup_edge_type[0])
                        ]["edge_index"].shape[1]
    print(
        f"num chg edges in both directions is equal: {num_chg_r == num_chg_l}")
    
def test_is_correct_p(dataset, sup_edge_type, p, total_num, prev_edges, is_train):
    # num_supervision divided by 2 because the same number of edges is generated as negative samples.
    # These are directed (i.e, a single link in one direction)
    num_supervision = dataset[sup_edge_type]["edge_label"].shape[0]
    if not is_train:
        num_supervision /=2

    
    num_msg = dataset[sup_edge_type
                      ]["edge_index"].shape[1]

    num = round(num_supervision + num_msg)
    expected_num = round(p*total_num + prev_edges)
    print(f"Is expected % of edges: {num == expected_num}")
    print(f"Expected {expected_num}, is {num}")

In [12]:
total_num_chg = data.data[supervision_edge_type]["edge_index"].shape[1]

datasets = {"train":train_data, "validation":val_data, "test":test_data}
percentage = [0.8,0.1,0.1]

prev_edges = 0
for name, p in zip(list(datasets.keys()), percentage):
    print(name + ":")
    test_equal_num_edges(datasets[name], supervision_edge_type)
    test_is_correct_p(datasets[name], supervision_edge_type, p, total_num_chg, prev_edges, name=="train")
    print("\n")

    num_sup_edges = datasets[name][supervision_edge_type]["edge_label"].shape[0]
    num_msg_edges = datasets[name][supervision_edge_type]["edge_index"].shape[1]
    if name !="train":
        num_sup_edges /= 2
        # no neg edges on train data, sampled on the fly while training

    prev_edges = round(num_sup_edges+num_msg_edges)

train:
num chg edges in both directions is equal: True
Is expected % of edges: True
Expected 30713, is 30713


validation:
num chg edges in both directions is equal: True
Is expected % of edges: True
Expected 34552, is 34552


test:
num chg edges in both directions is equal: True
Is expected % of edges: True
Expected 38391, is 38391




In [13]:
import yaml
configs_folder = "/biodata/nyanovsky/datasets/dti/best_models/"
with open(configs_folder+"sage_config.yaml","r") as file:
    sage_config = yaml.safe_load(file)

In [14]:
sage_config 

{'conv': {'aggr': 'max'},
 'gral': {'L2_norm': False,
  'batch_norm': False,
  'dropout': 0.0,
  'hidden_channels': 32,
  'layer_connectivity': 'False',
  'macro_aggregation': 'sum',
  'msg_passing_layers': 4,
  'normalize_output': False,
  'post_process_layers': 0,
  'pre_process_layers': 0},
 'train': {'delta': 0.1,
  'epochs': 500,
  'feature_dim': 128,
  'features': 'go2vec',
  'lr': 0.01,
  'patience': 10,
  'weight_decay': 0.001}}

In [15]:
sage_config["train"].pop("epochs")
sage_config["train"].pop("feature_dim")
sage_config["train"].pop("features")

'go2vec'

In [16]:
sage_config

{'conv': {'aggr': 'max'},
 'gral': {'L2_norm': False,
  'batch_norm': False,
  'dropout': 0.0,
  'hidden_channels': 32,
  'layer_connectivity': 'False',
  'macro_aggregation': 'sum',
  'msg_passing_layers': 4,
  'normalize_output': False,
  'post_process_layers': 0,
  'pre_process_layers': 0},
 'train': {'delta': 0.1, 'lr': 0.01, 'patience': 10, 'weight_decay': 0.001}}

Test model with single split

In [17]:
from model_class import LinkPredictor

model = LinkPredictor(conv_name="SAGE",
                      gral_params=sage_config["gral"],
                      conv_params=sage_config["conv"],
                      optimizer_params=sage_config["train"])

In [18]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
splits = {"train":train_data, "val":val_data, "test":test_data}
model.train(splits, sampler,supervision_edge_type, epochs=500)

{'train_loss': [0.6843099594116211,
  4.635708332061768,
  0.9613096714019775,
  0.707546591758728,
  0.6849134564399719,
  0.6995795369148254,
  0.680513322353363,
  0.669815182685852,
  0.6658753156661987,
  0.6519668698310852,
  0.6495211720466614,
  0.6246516108512878,
  0.6060360670089722,
  0.5776278376579285,
  0.5609737038612366,
  0.56667160987854,
  0.5319799184799194,
  0.5282571911811829,
  0.5186967253684998,
  0.5145243406295776,
  0.49914056062698364,
  0.4877128601074219,
  0.4825231730937958,
  0.4776168167591095,
  0.47365719079971313,
  0.4585306942462921,
  0.4534991979598999,
  0.44336098432540894,
  0.43758586049079895,
  0.4350123405456543,
  0.42401760816574097,
  0.42219868302345276,
  0.41818967461586,
  0.41503143310546875,
  0.40352264046669006,
  0.40424275398254395,
  0.39672577381134033,
  0.3937200903892517,
  0.38627246022224426,
  0.38997969031333923,
  0.37977492809295654,
  0.38504427671432495,
  0.3968793749809265,
  0.37249723076820374,
  0.3762813

In [23]:
model.evaluate(test_data)

{'loss': 0.34334754943847656, 'roc_auc': 0.948}

Test multiple splits

In [20]:
import numpy as np
aucs = []
for i in range(5):
    seed_everything(i)
    model.reset()
    train_data, val_data, test_data = data.split_data(task="link_prediction",
                                                  negative_sampler=sampler,
                                                  supervision_edge_type=supervision_edge_type,
                                                  disjoint_train_ratio=0.2, 
                                                  num_val=0.1,
                                                  num_test=0.1)
    splits = {"train":train_data, "val":val_data, "test":test_data}
    model.train(splits, sampler,supervision_edge_type, epochs=500)
    aucs.append(model.evaluate(test_data)["roc_auc"])

print(aucs)
print(np.mean(aucs))

[0.949, 0.949, 0.941, 0.948, 0.95]
0.9474
